# 04 · Build your own distributed circuit

The packaged primitives (`non_local_cnot`, `teleportation`, …) are just short
sequences of **circuit-builder** calls. This notebook shows how to drive that
builder yourself, in two steps:

1. **Re-derive the non-local CNOT from scratch** and check, line for line, that it
   equals the packaged one.
2. **Compose a new gadget** the library doesn't ship — a distributed logical
   **SWAP** made entirely from non-local CNOTs — and decode it.

`t.make_builder(code, num_blocks, p, ebits=...)` returns the low-level builder
(`CircuitBuilder` for BB codes, `CircuitBuilderSurface` for surface codes). Its
method vocabulary is summarised in `tmcbs/experiments.py`.


In [ ]:
# --- make the tmcbs package importable whether this notebook is launched from
# --- the package directory or from notebooks/ ------------------------------
import os, sys
for _cand in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), ".."))):
    if os.path.isdir(os.path.join(_cand, "tmcbs")) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import tmcbs as t
print("tmcbs version", t.__version__)

# Decoder used throughout. "teser" (Tesseract) is the decoder behind the paper's
# main results. If the compiled tesseract_decoder wheel will not load on your
# machine, switch to a pure-Python fallback: "BP-OSD" or "LSD".
DECODER = "teser"

P, P_EBIT = 1e-3, 1e-3
code = t.surface_code(5)        # try t.bb_code("[[18,4,4]] BB") too


## 1. The non-local CNOT, by hand

Block 0 is the control (on QC1), block 1 the target (on QC2). We:

* initialise both blocks and prepare `n` noisy ebits across the link,
* run a few syndrome rounds to settle the state,
* do the transversal-CNOT + measure + feed-forward dance that teleports the gate,
* run more syndrome rounds and read both blocks out (declaring detectors and one
  logical observable per block).


In [ ]:
b = t.make_builder(code, num_blocks=2, p=P, ebits=True)

b.initQubits(0)                                   # control block
b.initQubits(1)                                   # target block
b.prepareEbits(transError=P_EBIT)                 # n Bell pairs across the link
b.extractionRound([0, 1], firstPass=True)         # open detectors
for _ in range(3):
    b.extractionRound([0, 1], errors=True)        # settle

b.transversalOp("CX", [0, 0], typeArr=["BB", "e"])   # control block -> ebit reg 0
b.measureEbit0ThenCorrectEbit1()                     # measure + X feed-forward
b.transversalOp("CX", [1, 1], typeArr=["e", "BB"])   # ebit reg 1 -> target block
b.transversalOp("H",  [1], typeArr=["e"])
b.measureEbit1ThenCorrectCB(0)                       # measure + Z feed-forward

for _ in range(3):
    b.extractionRound([0, 1], errors=True)
b.measureDataQubits([0, 1])                          # logical readout
b.endOfCircuitDetectorsForLogicalMeasurementReadout(0)
b.endOfCircuitDetectorsForLogicalMeasurementReadout(1)
b.obsOffset(0, 0)                                    # observables for block 0 ...
b.obsOffset(1, code.k)                               # ... and block 1

my_cnot = b.getCirc()
assert str(my_cnot) == str(t.non_local_cnot(code, P, P_EBIT))
print("hand-built circuit is byte-identical to t.non_local_cnot  ✔")
print("detectors:", my_cnot.num_detectors, " observables:", my_cnot.num_observables)


## 2. A new gadget: a distributed logical SWAP

Now something the library doesn't provide. We build a logical **SWAP** between
two nodes using the standard decomposition
`SWAP(a, b) = CX(a, b); CX(b, a); CX(a, b)`, but every CNOT is the same
fresh-ebit non-local CNOT primitive used above.


In [ ]:
def nonlocal_cx(b, control, target, p_ebit=P_EBIT):
    """Append one transversal non-local CNOT (control -> target) using a fresh
    pair of ebit registers. Same primitive as inside t.non_local_cnot."""
    b.prepareEbits(transError=p_ebit)
    b.transversalOp("CX", [control, 0], typeArr=["BB", "e"])
    b.measureEbit0ThenCorrectEbit1()
    b.transversalOp("CX", [1, target], typeArr=["e", "BB"])
    b.transversalOp("H",  [1], typeArr=["e"])
    b.measureEbit1ThenCorrectCB(control)

def build_nonlocal_swap(code, p, p_ebit):
    b = t.make_builder(code, num_blocks=2, p=p, ebits=True)
    for blk in (0, 1):
        b.initQubits(blk)
    b.extractionRound([0, 1], firstPass=True)
    for _ in range(3):
        b.extractionRound([0, 1], errors=True)

    nonlocal_cx(b, 0, 1, p_ebit)            # first CX in the SWAP
    b.extractionRound([0, 1], errors=True)
    nonlocal_cx(b, 1, 0, p_ebit)            # reverse CX
    b.extractionRound([0, 1], errors=True)
    nonlocal_cx(b, 0, 1, p_ebit)            # final CX completes SWAP

    for _ in range(3):
        b.extractionRound([0, 1], errors=True)
    b.measureDataQubits([0, 1])
    b.endOfCircuitDetectorsForLogicalMeasurementReadout(0)
    b.endOfCircuitDetectorsForLogicalMeasurementReadout(1)
    b.obsOffset(0, 0)
    b.obsOffset(1, code.k)
    return b.getCirc()

swap = build_nonlocal_swap(code, P, P_EBIT)
print("SWAP circuit:", swap.num_qubits, "qubits,",
      swap.num_detectors, "detectors,", swap.num_observables, "observables")

# This should always succeed: the circuit's detectors and observables are deterministic.
_ = swap.detector_error_model()


The custom SWAP decodes like any other circuit in the library. As `p` shrinks,
the probability of reporting a wrong logical readout should fall:


In [ ]:
for p in (5e-3, 3e-3, 1e-3):
    circ = build_nonlocal_swap(code, p, p)
    r = t.estimate_ler(circ, p=p, d=code.d, decoder=DECODER, target_errors=30)
    print(f"  p={p:.0e}   SWAP {r}")


## Where to take it

* `t.parity_check_matrices(circ)` hands you `(chk, obs, priors)` to drive a custom
  decoder; `t.estimate_ler` works on *any* circuit you build.
* To add a gadget to the library, copy a constructor from `tmcbs/experiments.py`
  and re-order the builder calls — that is all the packaged primitives are.

* NOTE: including certain operations such as transversal Hadamards (those used in Teleportation are effectively physical) and creating more complex logical operations will require some care and knowledge of STIM.

